In [ ]:
!git clone https://github.com/elifsudayi/food-recipe-finder.git
%cd food-recipe-finder
!pip install -q -r requirements.txt
!mkdir -p results

import sys, json, re, time, pickle; sys.path.append(".")
import numpy as np, pandas as pd
import nltk; nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)

corpus = pd.read_csv("corpus/recipes.csv")
queries = pd.read_csv("corpus/queries.csv")
corpus["description"] = corpus["description"].fillna("")
corpus["diet_tags"] = corpus["diet_tags"].fillna("")
print(corpus.shape, queries.shape)

Cloning into 'food-recipe-finder'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 64 (delta 23), reused 52 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 169.19 KiB | 2.23 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/food-recipe-finder
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
(300, 8) (40, 6)


In [ ]:
from src.retrieval.text import TextProcessor
t = "Easy Tomatoes & Mushrooms, jalapeño omelettes cooked quickly"
for mode in ["raw", "none", "stem", "lemma"]:
    print(f"{mode:6s}", TextProcessor(mode)(t))

raw    ['easy', 'tomatoes', 'mushrooms', 'jalape', 'o', 'omelettes', 'cooked', 'quickly']
none   ['easy', 'tomatoes', 'mushrooms', 'jalapeno', 'omelettes', 'cooked', 'quickly']
stem   ['easi', 'tomato', 'mushroom', 'jalapeno', 'omelett', 'cook', 'quick']
lemma  ['easy', 'tomato', 'mushroom', 'jalapeno', 'omelette', 'cook', 'quickly']


In [ ]:
from src.retrieval.constraints import extract_constraints
ok, bad = 0, []
for _, q in queries.iterrows():
    c = extract_constraints(q["query"])
    m = re.search(r"minutes(<=|<)(\d+)", q["criteria"]); d = re.search(r"diet_tag=([a-z-]+)", q["criteria"])
    exp = (float(m.group(2)) if m else None, (m.group(1) == "<") if m else False, [d.group(1)] if d else [])
    got = (c.max_minutes, c.strict if c.max_minutes else False, c.diets)
    if exp == got: ok += 1
    else: bad.append((q["query"], exp, got))
print(f"constraint inference is correct: {ok}/{len(queries)}")
for b in bad: print(b)
print(extract_constraints("vegan french fries under 45 minutes"))

constraint inference is correct: 40/40
Constraints(max_minutes=45.0, strict=True, diets=['vegan'], text='french fries')


In [ ]:
from src.retrieval.tfidf import TfidfRetriever
from src.retrieval.metrics import evaluate

configs = [("tf-idf (raw tokens)", "raw", False),
           ("+ normalisation & stop-words", "none", False),
           ("+ stemming", "stem", False),
           ("+ lemmatization", "lemma", False),
           ("stemming + constraints [BASELINE]", "stem", True),
           ("lemmatization + constraints", "lemma", True)]
results, per_q_all = {}, {}
for name, mode, uc in configs:
    r = TfidfRetriever(corpus, mode, uc)
    s, by, per_q = evaluate(r.rank_ids, queries)
    results[name] = {"summary": s, "by_type": by.round(4).to_dict()}
    per_q_all[name] = per_q
    print(f"{name:38s} MRR {s['MRR']:.3f} | Recall@5 {s['Recall@5']:.3f}")

tf-idf (raw tokens)                    MRR 0.628 | Recall@5 0.445
+ normalisation & stop-words           MRR 0.594 | Recall@5 0.439
+ stemming                             MRR 0.703 | Recall@5 0.536
+ lemmatization                        MRR 0.703 | Recall@5 0.536
stemming + constraints [BASELINE]      MRR 0.946 | Recall@5 0.855
lemmatization + constraints            MRR 0.946 | Recall@5 0.860


In [ ]:
base_name = "stemming + constraints [BASELINE]"
print(pd.DataFrame(results[base_name]["by_type"]).round(3))

pq = per_q_all[base_name]
print("\nQueries with Recall@5 < 1:")
print(pq[pq["recall@5"] < 1][["query_id", "type", "query", "rr", "recall@5"]].to_string())

              MRR  recall@5
combo       1.000     1.000
diet        1.000     0.950
ingredient  0.900     0.740
semantic    0.883     0.780
time        1.000     0.925

Queries with Recall@5 < 1:
   query_id        type                                                           query        rr  recall@5
2       q03        time                           hamburger that takes less than 45 min  1.000000       0.6
6       q07        time                            pancakes that takes less than 20 min  1.000000       0.8
10      q11        diet                                             vegetarian omelette  1.000000       0.8
12      q13        diet                                              low carb hamburger  1.000000       0.8
16      q17  ingredient                                            pizza with mushrooms  1.000000       0.6
17      q18  ingredient                                              sushi with avocado  0.500000       0.6
18      q19  ingredient                         

In [ ]:
from src.utils import hardware_info
base = TfidfRetriever(corpus, "stem", True)

qs = queries["query"].tolist() * 5            # 200 calls
for q in qs[:10]: base.search(q, 5)           # warm-up
times = []
for q in qs:
    t0 = time.perf_counter(); base.search(q, 5); times.append((time.perf_counter() - t0) * 1000)
size_mb = len(pickle.dumps((base.vec.vocabulary_, base.vec.idf_, base.X))) / 1e6

out = {"model": "tfidf_stem_constraints", "summary": results[base_name]["summary"],
       "by_type": results[base_name]["by_type"],
       "size_mb": size_mb, "latency_cpu_ms": {"mean_ms": float(np.mean(times)), "std_ms": float(np.std(times)), "runs": len(times)},
       "ablation": {k: v["summary"] for k, v in results.items()},
       "constraint_extraction_exact_match": f"{ok}/{len(queries)}",
       "hardware": hardware_info()}
json.dump(out, open("results/retrieval_tfidf_metrics.json", "w"), indent=2)
print(json.dumps(out, indent=2))

{
  "model": "tfidf_stem_constraints",
  "summary": {
    "MRR": 0.9458333333333334,
    "Recall@5": 0.8550000000000001,
    "n_queries": 40
  },
  "by_type": {
    "MRR": {
      "combo": 1.0,
      "diet": 1.0,
      "ingredient": 0.9,
      "semantic": 0.8833,
      "time": 1.0
    },
    "recall@5": {
      "combo": 1.0,
      "diet": 0.95,
      "ingredient": 0.74,
      "semantic": 0.78,
      "time": 0.925
    }
  },
  "size_mb": 0.164714,
  "latency_cpu_ms": {
    "mean_ms": 1.1000381700046091,
    "std_ms": 0.38722552325842613,
    "runs": 200
  },
  "ablation": {
    "tf-idf (raw tokens)": {
      "MRR": 0.6282945526695527,
      "Recall@5": 0.4454166666666667,
      "n_queries": 40
    },
    "+ normalisation & stop-words": {
      "MRR": 0.5940683621933622,
      "Recall@5": 0.43916666666666665,
      "n_queries": 40
    },
    "+ stemming": {
      "MRR": 0.7031351981351982,
      "Recall@5": 0.5358333333333334,
      "n_queries": 40
    },
    "+ lemmatization": {
      "